In [1]:
import sys
sys.executable

'/home/gmvaz/Installs/miniforge3/envs/gems/bin/python'

In [2]:
# core
import numpy as np
import pandas as pd

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model

#iMATpy
from imatpy.imat import imat
from imatpy.model_utils import read_model
from imatpy.parse_gpr import gene_to_rxn_weights
from imatpy.model_creation import generate_model

In [3]:
# set solver to glpk, this is the default optimizer installed with cobra so it is the easiest one to set for imat since we it's already installed
Configuration().solver = "glpk"

In [4]:
# load the test salmonella model in cobrapy 
model = load_model("salmonella")
model

Name,iYS1720
Memory address,708c31fe9f50
Number of metabolites,2436
Number of reactions,3357
Number of genes,1707
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_1 - 1.0*BIOMASS_iRR1083_1_reverse_12350
Compartments,"cytosol, extracellular space, periplasm"


In [22]:
model.reactions[:4]

[<Reaction 12DGR120tipp at 0x708c2bd79650>,
 <Reaction EX_cobalt2_e at 0x708c2bd79610>,
 <Reaction 3OAR60 at 0x708c2bd79e50>,
 <Reaction 3OAR80 at 0x708c2bd7a590>]

In [23]:
model.metabolites[:4]

[<Metabolite 10fthf_c at 0x708c3acd7950>,
 <Metabolite 12dgr120_c at 0x708c2c6f1c90>,
 <Metabolite 12dgr140_c at 0x708c2c6f0710>,
 <Metabolite 12dgr141_c at 0x708c5dc22b50>]

In [24]:
model.genes[:4]

[<Gene STM3468 at 0x708c2bf74a50>,
 <Gene STM0320 at 0x708c31feab10>,
 <Gene STM0999 at 0x708c2bf75bd0>,
 <Gene STM1473 at 0x708c2bf75e10>]

In [28]:
# get gene IDs from our test model 
len(model.genes)
[g.id for g in model.genes][:10]

['STM3468',
 'STM0320',
 'STM0999',
 'STM1473',
 'STM2267',
 'STM2037',
 'STM4228',
 'STM4227',
 'STM4230',
 'STM4229']

In [26]:
# sanity check the gene id by seeing which reaction a gene maps
test_gene1 = model.genes.get_by_id("STM3468")
test_gene1.reactions

frozenset({<Reaction ACOTA at 0x708c2be5a110>,
           <Reaction SDPTA at 0x708c2b138290>})

In [27]:
# see the GPR associated with the reactions for that gene 
for r in test_gene1.reactions:
    print(r.id, "GPR:", r.gene_reaction_rule)

ACOTA GPR: STM3468 or STM1303
SDPTA GPR: s0001 or STM3468


In [29]:
# A pandas series of gene expression weights (fake)
# The gene IDs need to match the actual model.gene IDs
model_weights = pd.Series({
           "STM3468": 1,
           "STM0320": -1,
           "STM0999": -1,
           "STM1473": 1,
           "STM2267": -1,
           "STM2037": 1,
           "STM4228": -1,
           "STM4227": -1,
           "STM4230": 1,
           "STM4229": 1,
       })

In [32]:
# Reaction_weights=gene_to_rxn_weights(model=model, gene_weights=model_weights)

# Reindex list of gene IDs, keep the gene weights labeled in the pandas series, and additional added genes will get weight zero
gene_weights_all = model_weights.reindex([g.id for g in model.genes], fill_value=0)

# Get the reaction weights 
reaction_weights = gene_to_rxn_weights(model=model, gene_weights=gene_weights_all)

# most reactions have a weight of zero because the gene list defined above is very small
print(reaction_weights.head())

12DGR120tipp    0.0
EX_cobalt2_e    0.0
3OAR60          0.0
3OAR80          0.0
3OAS100         0.0
dtype: float64


In [43]:
# Sanity check reaction weights using ACOTA reaction as an examples
rxn_example = reaction_weights.loc["ACOTA"]
print(rxn_example)

# check why it is getting high expression
r = model.reactions.get_by_id("ACOTA")

print("GPR:", r.gene_reaction_rule)
print("Genes:", [g.id for g in r.genes])
print("Gene weights:")
gene_weights_all.loc[[g.id for g in r.genes]]

1.0
GPR: STM3468 or STM1303
Genes: ['STM1303', 'STM3468']
Gene weights:


STM1303    0
STM3468    1
dtype: int64

In [42]:
on = reaction_weights[reaction_weights == 1]
off = reaction_weights[reaction_weights == -1]
print(on)
print(off)

ACOTA    1.0
SDPTA    1.0
dtype: float64
ACNAMtex        -1.0
MALTHXabcpp     -1.0
MALTPTabcpp     -1.0
MALTTRabcpp     -1.0
MALTTTRabcpp    -1.0
MALTabcpp       -1.0
14GLUCANabcpp   -1.0
dtype: float64


In [44]:
# run imat
imat_results = imat(
    model=model,
    rxn_weights=reaction_weights,
    epsilon=1,
    threshold=0.01
)

In [34]:
# Print the imat objective
print(f"iMAT Objective: {imat_results.objective_value}")
print("Number of active reactions:",
      (imat_results.fluxes.abs() > 1e-6).sum())

iMAT Objective: 9.0
Number of active reactions: 97


In [35]:
# Print the imat flux distribution
print(f"iMAT Flux Distribution: \n{imat_results.fluxes}")

iMAT Flux Distribution: 
12DGR120tipp    0.000000e+00
EX_cobalt2_e   -0.000000e+00
3OAR60          0.000000e+00
3OAR80          0.000000e+00
3OAS100        -7.256389e-17
                    ...     
2AGPA141tipp   -6.873470e-16
2AGPA160tipp    2.775558e-17
2AGPA161tipp    0.000000e+00
2AGPA180tipp    0.000000e+00
2AGPA181tipp    0.000000e+00
Name: fluxes, Length: 3357, dtype: float64
